<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Data Carpentry · Python profesional</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Soluciones — Del notebook exploratorio al proyecto profesional**

Este laboratorio introduce una transición fundamental en ciencia de datos: pasar de un **notebook que simplemente funciona** a un trabajo que pueda ser **entendido, repetido, mantenido y reutilizado por otras personas**.

No se trata de dejar de utilizar notebooks. Los notebooks son herramientas excelentes para exploración, visualización, experimentación y comunicación. El objetivo es aprender **qué debe permanecer en el notebook y qué debería empezar a convertirse en código reusable**.

A lo largo del laboratorio construiremos un pequeño análisis de clientes, primero de forma exploratoria y después de forma progresivamente más profesional.

## Objetivos de aprendizaje

Al finalizar el laboratorio deberías ser capaz de:

1. Explicar qué diferencia un notebook exploratorio de un pipeline mantenible.
2. Reconocer señales de deuda técnica en notebooks.
3. Separar carga, validación, transformación, análisis y visualización.
4. Evitar dependencias ocultas entre celdas.
5. Diseñar transformaciones reproducibles.
6. Utilizar funciones para encapsular lógica.
7. Distinguir código exploratorio de código que debería vivir en `src/`.
8. Proponer una estructura básica para un proyecto de datos.
9. Evaluar si un notebook puede ser ejecutado de principio a fin.
10. Refactorizar un análisis sencillo sin cambiar sus resultados.

## Cómo trabajar con este notebook

**Versión de solución.** Contiene la misma parte guiada y una propuesta completa para los diez ejercicios finales, con comprobaciones y respuestas orientativas. Ejecuta todas las celdas en orden desde un entorno limpio. Las respuestas abiertas admiten alternativas justificadas.

**Tiempo:** selecciona con el docente los ejercicios que se resolverán durante la sesión; el resto puede utilizarse como ampliación.

**Entorno:** Python, NumPy, Pandas y Matplotlib; ejecutar desde la carpeta de esta sesión. Los ejemplos crean `proyecto_clientes_demo/` y la solución del ejercicio 9 crea `proyecto_productos_practica/`.


## 1. ¿Por qué los notebooks terminan siendo difíciles de mantener?

Un notebook suele comenzar de forma inocente:

- cargamos datos;
- hacemos una limpieza rápida;
- calculamos alguna variable;
- hacemos un gráfico;
- probamos otra idea;
- modificamos una celda anterior;
- añadimos otra transformación;
- copiamos código para probar una variante.

El problema aparece cuando el notebook deja de ser un experimento y empieza a convertirse en una pieza del proyecto.

### Síntomas habituales

- El notebook solo funciona si las celdas se ejecutan en un orden concreto.
- Hay variables globales cuyo origen no está claro.
- Existen varias versiones de la misma transformación.
- Las rutas de ficheros están escritas directamente en múltiples celdas.
- Los resultados cambian entre ejecuciones.
- Hay código copiado y pegado.
- Una celda hace demasiadas cosas.
- El notebook mezcla lógica de negocio, visualización y exportación.
- Nadie sabe qué celdas son imprescindibles y cuáles eran experimentos.

En este laboratorio veremos estos problemas de forma práctica.

## 2. Preparación del entorno

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print("Entorno preparado correctamente.")
print("Semilla:", SEED)

### ¿Por qué fijamos una semilla?

Cuando generamos datos aleatorios, una semilla permite que diferentes ejecuciones produzcan los mismos valores.

Esto mejora la **reproducibilidad**. Si dos estudiantes ejecutan el notebook con la misma semilla, deberían obtener los mismos datos y, por tanto, resultados equivalentes.

## 3. Creación de un dataset de trabajo

In [ ]:
n_clientes = 1200

clientes = pd.DataFrame({
    "cliente_id": np.arange(1, n_clientes + 1),
    "edad": rng.integers(18, 80, n_clientes),
    "ingresos": rng.normal(32000, 11500, n_clientes).clip(7000),
    "gasto_anual": rng.normal(14500, 6200, n_clientes).clip(700),
    "segmento": rng.choice(
        ["A", "B", "C"],
        n_clientes,
        p=[0.35, 0.45, 0.20]
    ),
    "canal": rng.choice(
        ["web", "tienda", "partner"],
        n_clientes,
        p=[0.50, 0.35, 0.15]
    )
})

clientes.head()

### Interpretación

Hemos creado una tabla sintética con 1.200 clientes. Cada fila representa una observación y las columnas recogen variables demográficas y de comportamiento.

Utilizar datos sintéticos aquí tiene dos ventajas:

- todo el laboratorio es autocontenido;
- no dependemos de un fichero externo para ejecutar la sesión.

En un proyecto real, esta celda sería sustituida por una función de carga desde CSV, base de datos, API o data warehouse.

In [ ]:
clientes.info()

### Interpretación

`DataFrame.info()` permite revisar rápidamente:

- número de filas;
- columnas disponibles;
- tipos de datos;
- valores nulos;
- memoria aproximada.

Esta comprobación es típicamente **exploratoria** y encaja muy bien dentro de un notebook.

In [ ]:
clientes.describe(include="all").T

### Interpretación

La tabla descriptiva permite detectar valores extraños y entender escalas.

Algunos ejemplos de preguntas que conviene hacerse:

- ¿Las edades están en un rango plausible?
- ¿Los ingresos contienen valores negativos?
- ¿Los segmentos aparecen con frecuencias razonables?
- ¿La escala de gasto es coherente con la de ingresos?

## 4. Primera exploración visual

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    clientes["ingresos"],
    clientes["gasto_anual"],
    alpha=0.35
)

ax.set_title("Relación entre ingresos y gasto anual")
ax.set_xlabel("Ingresos")
ax.set_ylabel("Gasto anual")

plt.show()

### Interpretación del gráfico

El diagrama de dispersión permite observar la relación entre ingresos y gasto.

No parece existir una asociación extremadamente fuerte. También pueden apreciarse clientes con combinaciones poco habituales, como ingresos bajos acompañados de gasto relativamente elevado.

En esta fase el gráfico sirve para **formular preguntas**, no para establecer conclusiones causales.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

clientes["segmento"].value_counts().sort_index().plot(
    kind="bar",
    ax=ax
)

ax.set_title("Número de clientes por segmento")
ax.set_xlabel("Segmento")
ax.set_ylabel("Número de clientes")
plt.xticks(rotation=0)

plt.show()

### Interpretación del gráfico

El segmento B es el más numeroso porque así lo definimos al generar los datos.

Esto también ilustra una buena práctica: antes de interpretar un gráfico conviene saber **cómo fueron obtenidos los datos**. Un patrón puede provenir del proceso generador y no de un fenómeno real.

# 5. Un notebook que empieza a degradarse

Vamos a escribir ahora código funcional pero poco mantenible.

El objetivo no es afirmar que este código sea siempre incorrecto. El objetivo es identificar por qué puede convertirse en un problema cuando crece.

In [ ]:
# Versión exploratoria y poco mantenible

resultado = []

for i in range(len(clientes)):
    ingresos = clientes.iloc[i]["ingresos"]
    gasto = clientes.iloc[i]["gasto_anual"]

    if ingresos > 35000 and gasto < 16000:
        resultado.append("potencial")
    else:
        resultado.append("normal")

clientes["perfil_v1"] = resultado

clientes[["cliente_id", "ingresos", "gasto_anual", "perfil_v1"]].head(10)

### ¿Qué problemas tiene este bloque?

Funciona, pero presenta varias señales de deuda técnica:

1. Los umbrales `35000` y `16000` son números mágicos.
2. La regla de negocio está escondida dentro del bucle.
3. Utilizamos un bucle Python para una operación que puede expresarse vectorialmente.
4. La columna creada se llama `perfil_v1`, un nombre que sugiere que habrá versiones posteriores.
5. Si la regla debe utilizarse en otro notebook tendremos que copiarla.
6. No existe una validación previa de columnas.
7. No hay explicación formal de qué significa `"potencial"`.

## 6. Dependencias ocultas entre celdas

In [ ]:
umbral_ingresos = 35000
umbral_gasto = 16000

In [ ]:
# Esta celda depende de variables creadas anteriormente.

clientes["perfil_dependiente"] = np.where(
    (clientes["ingresos"] > umbral_ingresos)
    & (clientes["gasto_anual"] < umbral_gasto),
    "potencial",
    "normal"
)

clientes["perfil_dependiente"].value_counts()

### ¿Qué ocurriría si ejecutamos solo la segunda celda?

Fallaría si `umbral_ingresos` o `umbral_gasto` no existieran en memoria.

Este es uno de los problemas más frecuentes de los notebooks: el estado del kernel puede ocultar dependencias.

### Regla práctica

Un notebook destinado a otras personas debería poder ejecutarse con:

**Kernel limpio → Run All**

sin depender de estados anteriores.

## 7. Primer paso de refactorización: encapsular la lógica

In [ ]:
def clasificar_cliente(
    ingresos: float,
    gasto_anual: float,
    umbral_ingresos: float = 35000,
    umbral_gasto: float = 16000
) -> str:
    if ingresos > umbral_ingresos and gasto_anual < umbral_gasto:
        return "potencial"
    return "normal"

In [ ]:
clasificar_cliente(
    ingresos=42000,
    gasto_anual=12000
)

### Interpretación

Ahora la regla está aislada.

Esto aporta varias ventajas:

- tiene un nombre;
- puede reutilizarse;
- puede probarse;
- los umbrales son explícitos;
- modificar la lógica ya no requiere buscarla en múltiples celdas.

In [ ]:
clientes["perfil_funcion"] = [
    clasificar_cliente(ingresos, gasto)
    for ingresos, gasto in zip(
        clientes["ingresos"],
        clientes["gasto_anual"]
    )
]

clientes["perfil_funcion"].value_counts()

## 8. Segundo paso: pensar en operaciones vectorizadas

In [ ]:
clientes["perfil_vectorizado"] = np.where(
    (clientes["ingresos"] > 35000)
    & (clientes["gasto_anual"] < 16000),
    "potencial",
    "normal"
)

clientes["perfil_vectorizado"].value_counts()

### Función fila a fila vs vectorización

Para lógica compleja, una función puede ser muy clara.

Para reglas simples aplicadas a grandes DataFrames, Pandas/NumPy suele permitir una expresión vectorizada más eficiente.

Ambas decisiones pueden ser profesionales. La pregunta correcta no es “¿qué técnica es siempre mejor?”, sino:

- ¿cuál expresa mejor la intención?
- ¿cuál es suficientemente eficiente?
- ¿cuál es más fácil de mantener?

## 9. Comparación de rendimiento

In [ ]:
clientes_benchmark = pd.concat(
    [clientes[["ingresos", "gasto_anual"]]] * 100,
    ignore_index=True
)

len(clientes_benchmark)

In [ ]:
inicio = time.perf_counter()

_ = [
    clasificar_cliente(i, g)
    for i, g in zip(
        clientes_benchmark["ingresos"],
        clientes_benchmark["gasto_anual"]
    )
]

tiempo_funcion = time.perf_counter() - inicio


inicio = time.perf_counter()

_ = np.where(
    (clientes_benchmark["ingresos"] > 35000)
    & (clientes_benchmark["gasto_anual"] < 16000),
    "potencial",
    "normal"
)

tiempo_vectorizado = time.perf_counter() - inicio

comparacion_tiempos = pd.Series({
    "Función fila a fila": tiempo_funcion,
    "Vectorizado": tiempo_vectorizado
})

comparacion_tiempos

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

comparacion_tiempos.plot(kind="bar", ax=ax)

ax.set_title("Comparación de tiempo de ejecución")
ax.set_ylabel("Segundos")
ax.set_xlabel("")
plt.xticks(rotation=0)

plt.show()

### Interpretación del gráfico

La versión vectorizada debería ser claramente más rápida para un dataset grande.

Esto ocurre porque Pandas y NumPy ejecutan gran parte de estas operaciones en código optimizado de bajo nivel, evitando el coste de iterar fila a fila desde Python.

La lección no es que *todo* deba vectorizarse, sino que conviene reconocer cuándo estamos utilizando Python como motor de iteración sobre estructuras diseñadas precisamente para evitarlo.

## 10. Separar transformación de análisis

In [ ]:
def preparar_clientes(datos: pd.DataFrame) -> pd.DataFrame:
    salida = datos.copy()

    salida["ratio_gasto"] = (
        salida["gasto_anual"] / salida["ingresos"]
    )

    salida["alto_consumo"] = (
        salida["ratio_gasto"] > 0.60
    )

    return salida

In [ ]:
clientes_preparados = preparar_clientes(clientes)

clientes_preparados[
    [
        "cliente_id",
        "ingresos",
        "gasto_anual",
        "ratio_gasto",
        "alto_consumo"
    ]
].head()

### ¿Por qué hacemos una copia?

La función recibe un DataFrame y devuelve otro transformado.

Al utilizar `.copy()` evitamos modificar silenciosamente el objeto original recibido.

Este estilo reduce efectos secundarios y facilita entender qué función transforma cada objeto.

## 11. Separar análisis de transformación

In [ ]:
def resumen_por_segmento(datos: pd.DataFrame) -> pd.DataFrame:
    return (
        datos
        .groupby("segmento")
        .agg(
            clientes=("cliente_id", "size"),
            ingreso_medio=("ingresos", "mean"),
            gasto_medio=("gasto_anual", "mean"),
            ratio_medio=("ratio_gasto", "mean"),
            pct_alto_consumo=("alto_consumo", "mean")
        )
        .reset_index()
    )

In [ ]:
resumen_segmentos = resumen_por_segmento(clientes_preparados)
resumen_segmentos

### Interpretación

Tenemos ahora dos responsabilidades distintas:

- `preparar_clientes()` crea variables derivadas.
- `resumen_por_segmento()` produce una tabla analítica.

Esta separación facilita:

- probar cada etapa;
- reutilizar transformaciones;
- cambiar el resumen sin tocar la preparación;
- cambiar la preparación sin reescribir la visualización.

## 12. Visualización a partir de resultados ya preparados

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    resumen_segmentos["segmento"],
    resumen_segmentos["pct_alto_consumo"]
)

ax.set_title("Proporción de clientes de alto consumo")
ax.set_xlabel("Segmento")
ax.set_ylabel("Proporción")

plt.show()

### Interpretación del gráfico

El gráfico representa la proporción de clientes cuyo gasto anual supera el 60 % de sus ingresos.

La visualización consume una tabla ya preparada. No necesita saber cómo se calculó `ratio_gasto` ni cómo se definió `alto_consumo`.

Esta separación es importante porque la visualización debería trabajar sobre **resultados analíticos**, no convertirse en el lugar donde vive la lógica del pipeline.

## 13. Un segundo gráfico: distribución del ratio de gasto

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for segmento in sorted(clientes_preparados["segmento"].unique()):
    datos_segmento = clientes_preparados.loc[
        clientes_preparados["segmento"] == segmento,
        "ratio_gasto"
    ]

    ax.hist(
        datos_segmento,
        bins=35,
        alpha=0.45,
        label=f"Segmento {segmento}"
    )

ax.set_title("Distribución del ratio gasto / ingresos")
ax.set_xlabel("Ratio gasto / ingresos")
ax.set_ylabel("Frecuencia")
ax.legend()

plt.show()

### Interpretación del gráfico

Las distribuciones se solapan considerablemente, lo que sugiere que los segmentos sintéticos no presentan diferencias extremadamente marcadas en este indicador.

Esto es coherente con la generación de datos: `segmento` se asignó independientemente de ingresos y gasto.

Este detalle es didácticamente importante: un buen análisis debe interpretar los resultados **en función del proceso de generación o procedencia de los datos**.

# 14. Calidad de datos: detectar antes de corregir

Hasta ahora nuestro dataset era demasiado limpio.

En un proyecto real, una de las tareas más importantes consiste en encontrar problemas como:

- valores ausentes;
- duplicados;
- valores imposibles;
- outliers;
- categorías inesperadas.

Hay una regla metodológica importante:

> **primero observamos y medimos la inconsistencia; después decidimos cómo corregirla.**

Si limpiamos los datos inmediatamente, ocultamos el problema y perdemos información sobre su origen.

## 14.1 Crear una copia con inconsistencias deliberadas

Para aprender el proceso, introduciremos varios problemas de forma controlada en una copia del dataset.
El dataset original permanecerá intacto.

In [ ]:
clientes_sucios = clientes.copy()

# 1. Valores ausentes
clientes_sucios.loc[[5, 42, 300], "ingresos"] = np.nan
clientes_sucios.loc[[17, 91], "segmento"] = np.nan

# 2. Valores imposibles
clientes_sucios.loc[25, "edad"] = -4
clientes_sucios.loc[88, "gasto_anual"] = -1200

# 3. Categoría inesperada
clientes_sucios.loc[150, "canal"] = "desconocido"

# 4. Outlier extremo
clientes_sucios.loc[220, "ingresos"] = 500_000

# 5. Registro duplicado
clientes_sucios = pd.concat(
    [clientes_sucios, clientes_sucios.iloc[[10]]],
    ignore_index=True
)

clientes_sucios.shape

## 14.2 Primera inspección: no corregimos nada todavía

Antes de modificar los datos vamos a comprobar qué problemas existen.

In [ ]:
resumen_calidad = pd.DataFrame({
    "nulos": clientes_sucios.isna().sum(),
    "tipo": clientes_sucios.dtypes.astype(str)
})

resumen_calidad

### Interpretación

Ya podemos observar valores ausentes en algunas variables.

Pero una tabla de nulos no detecta todos los problemas: por ejemplo, una edad de `-4` no es nula,
aunque conceptualmente es inválida.

In [ ]:
print("Duplicados completos:", clientes_sucios.duplicated().sum())
print("Edades negativas:", (clientes_sucios["edad"] < 0).sum())
print("Gastos negativos:", (clientes_sucios["gasto_anual"] < 0).sum())

canales_validos = {"web", "tienda", "partner"}

canales_invalidos = ~clientes_sucios["canal"].isin(canales_validos)

print("Canales inesperados:", canales_invalidos.sum())
print(
    clientes_sucios.loc[
        canales_invalidos,
        ["cliente_id", "canal"]
    ]
)

## 14.3 Visualizar una inconsistencia antes de corregirla

El outlier de ingresos puede ser difícil de apreciar únicamente mirando tablas.
Vamos a visualizar la distribución **antes** de aplicar ninguna corrección.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.boxplot(
    clientes_sucios["ingresos"].dropna(),
    vert=False
)

ax.set_title("Ingresos ANTES de limpiar inconsistencias")
ax.set_xlabel("Ingresos")

plt.show()

### Interpretación

El valor cercano a 500.000 domina la escala del gráfico.

Aquí todavía no hemos decidido si se trata de:

- un error de captura;
- un cliente real excepcional;
- una unidad monetaria incorrecta;
- un problema de integración.

**Un outlier no debe eliminarse automáticamente solo por ser extremo.**

Primero se detecta y después se investiga o se aplica una regla justificada.

In [ ]:
# Localizamos los registros extremos para poder inspeccionarlos

clientes_sucios.loc[
    clientes_sucios["ingresos"] > 150_000,
    ["cliente_id", "edad", "ingresos", "gasto_anual", "segmento", "canal"]
]

## 14.4 Construir una función de diagnóstico

Una buena práctica es encapsular las comprobaciones de calidad en una función.
Esta función **no corrige** todavía: únicamente diagnostica.

In [ ]:
def diagnosticar_calidad_clientes(
    datos: pd.DataFrame
) -> pd.Series:

    canales_validos = {"web", "tienda", "partner"}

    return pd.Series({
        "filas": len(datos),
        "duplicados": datos.duplicated().sum(),
        "nulos_totales": datos.isna().sum().sum(),
        "edades_negativas": (datos["edad"] < 0).sum(),
        "gastos_negativos": (datos["gasto_anual"] < 0).sum(),
        "canales_invalidos": (
            ~datos["canal"].isin(canales_validos)
        ).sum(),
        "ingresos_mayores_150k": (
            datos["ingresos"] > 150_000
        ).sum()
    })


diagnostico_antes = diagnosticar_calidad_clientes(
    clientes_sucios
)

diagnostico_antes

## 14.5 Ahora sí: aplicar reglas de limpieza

Vamos a definir reglas didácticas explícitas.

En un proyecto real estas decisiones deberían estar sustentadas por conocimiento del dominio.

In [ ]:
def limpiar_clientes(
    datos: pd.DataFrame
) -> pd.DataFrame:

    salida = datos.copy()

    # 1. Eliminar duplicados completos
    salida = salida.drop_duplicates()

    # 2. Edades negativas -> valor ausente
    salida.loc[
        salida["edad"] < 0,
        "edad"
    ] = np.nan

    # 3. Gastos negativos -> valor ausente
    salida.loc[
        salida["gasto_anual"] < 0,
        "gasto_anual"
    ] = np.nan

    # 4. Categorías inesperadas -> valor ausente
    canales_validos = {"web", "tienda", "partner"}

    salida.loc[
        ~salida["canal"].isin(canales_validos),
        "canal"
    ] = np.nan

    # 5. Imputación simple de numéricos con mediana
    salida["edad"] = salida["edad"].fillna(
        salida["edad"].median()
    )

    salida["ingresos"] = salida["ingresos"].fillna(
        salida["ingresos"].median()
    )

    salida["gasto_anual"] = salida["gasto_anual"].fillna(
        salida["gasto_anual"].median()
    )

    # 6. Imputación categórica con moda
    salida["segmento"] = salida["segmento"].fillna(
        salida["segmento"].mode().iloc[0]
    )

    salida["canal"] = salida["canal"].fillna(
        salida["canal"].mode().iloc[0]
    )

    # 7. Para este laboratorio tratamos >150k como error
    # y lo sustituimos por la mediana.
    mediana_ingresos = salida.loc[
        salida["ingresos"] <= 150_000,
        "ingresos"
    ].median()

    salida.loc[
        salida["ingresos"] > 150_000,
        "ingresos"
    ] = mediana_ingresos

    return salida

In [ ]:
clientes_limpios = limpiar_clientes(
    clientes_sucios
)

clientes_limpios.head()

## 14.6 Repetir exactamente el diagnóstico después de limpiar

Esta etapa es fundamental.

No debemos asumir que una función de limpieza ha funcionado simplemente porque no ha generado errores.

In [ ]:
diagnostico_despues = diagnosticar_calidad_clientes(
    clientes_limpios
)

comparacion_calidad = pd.DataFrame({
    "antes": diagnostico_antes,
    "despues": diagnostico_despues
})

comparacion_calidad

### Interpretación

Ahora podemos comprobar cuantitativamente qué problemas han desaparecido.

Esto convierte la limpieza en un proceso **auditable**:

1. detectamos;
2. cuantificamos;
3. corregimos;
4. volvemos a medir.

## 14.7 Visualización antes vs después

Vamos a repetir el mismo gráfico para comprobar visualmente el efecto de la corrección.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.boxplot(
    [
        clientes_sucios["ingresos"].dropna(),
        clientes_limpios["ingresos"].dropna()
    ],
    vert=False,
    tick_labels=["Antes", "Después"]
)

ax.set_title("Ingresos: antes y después de la limpieza")
ax.set_xlabel("Ingresos")

plt.show()

### Interpretación del gráfico

La escala vuelve a representar de forma útil la mayoría de observaciones después de tratar el valor extremo.

Sin embargo, debemos recordar que el cambio solo es correcto porque **hemos definido explícitamente que ese valor era un error para este ejercicio**.

En datos reales, una corrección semejante requeriría una regla de negocio, trazabilidad y, posiblemente, consulta con el propietario del dato.

## 14.8 Visualizar valores ausentes antes y después

Una visualización sencilla de los conteos de nulos ayuda a comprobar dónde se concentraban los problemas.

In [ ]:
nulos_comparacion = pd.DataFrame({
    "Antes": clientes_sucios.isna().sum(),
    "Después": clientes_limpios.isna().sum()
})

nulos_comparacion

In [ ]:
ax = nulos_comparacion.plot(
    kind="bar",
    figsize=(9, 4)
)

ax.set_title("Valores ausentes por variable")
ax.set_xlabel("Variable")
ax.set_ylabel("Número de valores ausentes")

plt.xticks(rotation=45)

plt.show()

### Interpretación

La gráfica permite identificar rápidamente qué columnas presentaban incompletitud y comprobar que,
después de las reglas aplicadas en este laboratorio, no quedan valores ausentes.

La visualización complementa la validación numérica, pero no la sustituye.

## 15. Validar antes de transformar

In [ ]:
def validar_clientes(datos: pd.DataFrame) -> None:
    columnas_requeridas = {
        "cliente_id",
        "edad",
        "ingresos",
        "gasto_anual",
        "segmento",
        "canal"
    }

    faltantes = columnas_requeridas - set(datos.columns)

    if faltantes:
        raise ValueError(
            f"Faltan columnas requeridas: {sorted(faltantes)}"
        )

    if (datos["ingresos"] <= 0).any():
        raise ValueError(
            "Se han detectado ingresos no positivos."
        )

    if (datos["gasto_anual"] < 0).any():
        raise ValueError(
            "Se han detectado valores negativos en gasto_anual."
        )

In [ ]:
validar_clientes(clientes)
print("Validación superada.")

### ¿Por qué validar?

Sin validación, un pipeline puede producir resultados técnicamente correctos sobre datos conceptualmente incorrectos.

Ejemplo:

```python
ratio = gasto / ingresos
```

es una operación válida para Python incluso aunque `ingresos` contenga valores absurdos.

El código profesional debe proteger algunas de sus **suposiciones mínimas**.

## 16. Construcción de un pipeline sencillo

In [ ]:
def ejecutar_pipeline_clientes(
    datos: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:

    validar_clientes(datos)

    preparados = preparar_clientes(datos)

    resumen = resumen_por_segmento(preparados)

    return preparados, resumen

In [ ]:
# El pipeline se ejecuta sobre los datos ya limpiados y validados.
clientes_finales, resumen_final = ejecutar_pipeline_clientes(
    clientes_limpios
)

resumen_final

### Lo que acabamos de conseguir

Nuestro flujo tiene ahora una secuencia explícita:

**datos → validación → preparación → análisis → visualización**

Esto ya se aproxima más a un pipeline.

Todavía estamos dentro de un notebook, lo cual es perfectamente válido para aprender y experimentar. El siguiente paso sería mover las funciones estables a módulos Python.

## 17. ¿Qué debería quedarse en el notebook?

Una división razonable sería:

### Notebook

- pregunta analítica;
- explicaciones;
- exploración;
- visualizaciones;
- interpretación;
- decisiones del analista;
- comparación de alternativas.

### Código reusable (`src/`)

- carga de datos;
- validaciones;
- limpieza;
- creación de features;
- reglas de negocio;
- funciones reutilizadas por varios notebooks;
- exportación;
- utilidades.

La frontera no es absoluta. Lo importante es evitar que la lógica crítica del proyecto quede escondida en celdas difíciles de localizar.

## 18. Crear automáticamente la estructura de un proyecto de datos

Hasta ahora hemos hablado de la estructura de carpetas de forma conceptual.

En un proyecto real no queremos limitarnos a copiar y pegar una estructura en el README:
podemos **crear la estructura automáticamente desde Python**.

Esto tiene varias ventajas:

- evita errores al crear carpetas manualmente;
- permite inicializar nuevos proyectos de forma reproducible;
- garantiza que todo el alumnado parte de la misma estructura;
- introduce el uso de `pathlib`, muy recomendable frente a concatenar rutas como strings.

In [ ]:
from pathlib import Path

raiz_proyecto = Path("proyecto_clientes_demo")

carpetas = [
    raiz_proyecto / "data" / "raw",
    raiz_proyecto / "data" / "processed",
    raiz_proyecto / "notebooks",
    raiz_proyecto / "src" / "proyecto_clientes",
    raiz_proyecto / "tests",
]

for carpeta in carpetas:
    carpeta.mkdir(parents=True, exist_ok=True)

print("Estructura de carpetas creada correctamente.")

### ¿Qué hace `mkdir(parents=True, exist_ok=True)`?

- `parents=True` crea también las carpetas padre necesarias.
- `exist_ok=True` evita un error si la carpeta ya existe.

Esto hace que la celda sea **idempotente**: podemos ejecutarla varias veces sin romper el notebook.

In [ ]:
# Creamos también algunos ficheros mínimos de proyecto

ficheros_iniciales = {
    raiz_proyecto / "src" / "proyecto_clientes" / "__init__.py": "",
    raiz_proyecto / "README.md": "# Proyecto de clientes\n",
    raiz_proyecto / ".gitignore": ".venv/\n__pycache__/\n.ipynb_checkpoints/\n",
    raiz_proyecto / "tests" / "__init__.py": "",
}

for ruta, contenido in ficheros_iniciales.items():
    ruta.write_text(contenido, encoding="utf-8")

print("Ficheros mínimos creados.")

### Verificar que la estructura se ha creado realmente

No debemos asumir que una operación sobre el sistema de ficheros ha funcionado.
Vamos a inspeccionar el resultado.

In [ ]:
for ruta in sorted(raiz_proyecto.rglob("*")):
    nivel = len(ruta.relative_to(raiz_proyecto).parts) - 1
    sangria = "    " * nivel
    simbolo = "📁" if ruta.is_dir() else "📄"
    print(f"{sangria}{simbolo} {ruta.name}")

### Interpretación

Ahora la estructura no es una representación teórica: existe físicamente en el entorno de ejecución.

Más adelante podríamos guardar:

- los CSV originales en `data/raw/`;
- datos transformados en `data/processed/`;
- funciones reutilizables en `src/proyecto_clientes/`;
- pruebas automáticas en `tests/`.

Este tipo de automatización es el primer paso hacia plantillas de proyecto, scripts de inicialización o herramientas como Cookiecutter.

In [ ]:
# Comprobación programática de que las carpetas esenciales existen

carpetas_obligatorias = [
    raiz_proyecto / "data" / "raw",
    raiz_proyecto / "data" / "processed",
    raiz_proyecto / "notebooks",
    raiz_proyecto / "src" / "proyecto_clientes",
    raiz_proyecto / "tests",
]

estado_carpetas = {
    str(carpeta): carpeta.exists()
    for carpeta in carpetas_obligatorias
}

estado_carpetas

**Interpretación.** Todas las entradas deberían devolver `True`.

Esta forma de comprobación es más robusta que mirar únicamente la salida impresa:
estamos **validando automáticamente** el estado del proyecto.

## 19. Cómo se vería la separación en módulos

In [ ]:
validation_py = '''
def validar_clientes(datos):
    ...
'''

features_py = '''
def preparar_clientes(datos):
    ...
'''

analysis_py = '''
def resumen_por_segmento(datos):
    ...
'''

print("validation.py")
print(validation_py)

print("features.py")
print(features_py)

print("analysis.py")
print(analysis_py)

En un proyecto real, el notebook podría importar estas funciones:

```python
from proyecto_clientes.validation import validar_clientes
from proyecto_clientes.features import preparar_clientes
from proyecto_clientes.analysis import resumen_por_segmento
```

El notebook quedaría centrado en análisis y comunicación, mientras que las transformaciones estables vivirían en el paquete.

# 20. Mini-práctica guiada — Refactorización progresiva

Vamos a trabajar ahora con un bloque deliberadamente poco profesional.

### Código original

El siguiente bloque:

- modifica directamente el DataFrame;
- contiene números mágicos;
- mezcla transformación y clasificación;
- no valida;
- no tiene funciones reutilizables.

In [ ]:
ventas = pd.DataFrame({
    "venta_id": np.arange(1, 801),
    "importe": rng.gamma(2, 120, 800),
    "unidades": rng.integers(1, 8, 800),
    "canal": rng.choice(
        ["web", "tienda", "partner"],
        800
    )
})

ventas.head()

In [ ]:
# Código deliberadamente mejorable

ventas["ticket_unitario"] = (
    ventas["importe"] / ventas["unidades"]
)

ventas["tipo_venta"] = "normal"

ventas.loc[
    ventas["importe"] > 400,
    "tipo_venta"
] = "alta"

ventas.loc[
    ventas["importe"] > 700,
    "tipo_venta"
] = "muy_alta"

resumen_ventas = ventas.groupby("canal")[
    "importe"
].mean()

resumen_ventas

### Paso 1 — Extraer preparación

In [ ]:
def preparar_ventas(datos: pd.DataFrame) -> pd.DataFrame:
    salida = datos.copy()

    salida["ticket_unitario"] = (
        salida["importe"] / salida["unidades"]
    )

    return salida

### Paso 2 — Extraer clasificación

In [ ]:
def clasificar_ventas(
    datos: pd.DataFrame,
    umbral_alta: float = 400,
    umbral_muy_alta: float = 700
) -> pd.DataFrame:

    salida = datos.copy()

    condiciones = [
        salida["importe"] > umbral_muy_alta,
        salida["importe"] > umbral_alta
    ]

    categorias = [
        "muy_alta",
        "alta"
    ]

    salida["tipo_venta"] = np.select(
        condiciones,
        categorias,
        default="normal"
    )

    return salida

### Paso 3 — Extraer análisis

In [ ]:
def resumen_ventas_por_canal(
    datos: pd.DataFrame
) -> pd.DataFrame:

    return (
        datos
        .groupby("canal")
        .agg(
            operaciones=("venta_id", "size"),
            importe_medio=("importe", "mean"),
            ticket_unitario_medio=("ticket_unitario", "mean")
        )
        .reset_index()
    )

### Paso 4 — Ejecutar el pipeline refactorizado

In [ ]:
ventas_preparadas = preparar_ventas(ventas)

ventas_clasificadas = clasificar_ventas(
    ventas_preparadas
)

resumen_ventas_refactorizado = (
    resumen_ventas_por_canal(
        ventas_clasificadas
    )
)

resumen_ventas_refactorizado

### Paso 5 — Visualizar

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    resumen_ventas_refactorizado["canal"],
    resumen_ventas_refactorizado["importe_medio"]
)

ax.set_title("Importe medio por canal")
ax.set_xlabel("Canal")
ax.set_ylabel("Importe medio")

plt.show()

### Interpretación

Ahora cada etapa tiene una responsabilidad:

- `preparar_ventas()` crea variables derivadas.
- `clasificar_ventas()` implementa una regla.
- `resumen_ventas_por_canal()` produce resultados analíticos.
- el notebook visualiza e interpreta.

Si mañana cambian los umbrales de clasificación, solo habría que modificar o parametrizar una función.

# 21. Comprobación de equivalencia

In [ ]:
comparacion = pd.DataFrame({
    "original": resumen_ventas.sort_index(),
    "refactorizado": (
        resumen_ventas_refactorizado
        .set_index("canal")["importe_medio"]
        .sort_index()
    )
})

comparacion["diferencia"] = (
    comparacion["original"]
    - comparacion["refactorizado"]
)

comparacion

### Interpretación

Refactorizar conserva el resultado mientras mejora la estructura. La columna `diferencia` debe contener ceros o valores numéricamente próximos a cero; los valores ausentes no demuestran equivalencia.


# 22. Checklist de calidad de un notebook

Antes de compartir un notebook, conviene revisar:

- [ ] ¿Se ejecuta desde un kernel limpio?
- [ ] ¿Las celdas están en un orden lógico?
- [ ] ¿La semilla está fijada cuando existe aleatoriedad?
- [ ] ¿Las rutas están centralizadas?
- [ ] ¿Los nombres explican qué contienen las variables?
- [ ] ¿Las transformaciones repetidas se han convertido en funciones?
- [ ] ¿Las reglas importantes están documentadas?
- [ ] ¿La visualización está separada de la transformación?
- [ ] ¿Los resultados incluyen interpretación?
- [ ] ¿El código reusable debería trasladarse a `src/`?

# 23. Resumen del laboratorio

En este laboratorio hemos pasado por tres niveles:

### Nivel 1 — Exploración

Código rápido para comprender los datos.

### Nivel 2 — Notebook estructurado

Funciones, orden lógico, validaciones y transformaciones reproducibles.

### Nivel 3 — Proyecto profesional

Las funciones estables empiezan a trasladarse a módulos importables y testeables.

El objetivo no es eliminar los notebooks, sino utilizarlos para aquello en lo que son especialmente buenos: **explorar, comunicar e interpretar**.

# 24. Ejercicios finales

Los ejercicios 1–5 consolidan la estructura y la validación. Los ejercicios 6–9 son ampliaciones; el 10 integra el recorrido completo.


## Ejercicio 1 — Detectar problemas

Revisa el bloque siguiente. Identifica tres dependencias entre celdas, tres nombres mejorables, dos bloques duplicados, dos números mágicos y una transformación que convertirías en función. No hace falta ejecutar el bloque: razona sobre él.


### Bloque para revisar

```python
# Celda A
d = datos_entrada.copy()
# Celda B
x = d["unidades"] * d["precio"]
y = x * (1 - 0.10)
# Celda C
x2 = d["unidades"] * d["precio"]
y2 = x2 * (1 - 0.10)
d["importe_neto"] = y
d["tipo"] = np.select([y > 700, y > 400], ["muy_alta", "alta"], default="normal")
# Celda D
r = d.groupby("producto")["importe_neto"].sum()
```


### Solución razonada

Una respuesta posible:

- Dependencias: `d` requiere `datos_entrada`; `x` requiere `d`; `y` requiere `x`; `r` requiere `d` y su columna derivada.
- Nombres: `d` → `ventas`; `x` → `importe_bruto`; `y` → `importe_neto`; `r` → `ventas_por_producto`.
- Duplicación: el cálculo de bruto se repite en `x` y `x2`; el cálculo de neto se repite en `y` y `y2`.
- Números mágicos: `400` y `700` son umbrales de negocio sin nombre; el descuento `0.10` se repite y debería ser configurable.
- Transformación reusable: cálculo de bruto y neto, trabajando sobre una copia del DataFrame.
- Además, el bloque modifica `d`, mezcla preparación con análisis y no valida las entradas.


## Ejercicio 2 — Refactorización simple

Crea un DataFrame de productos con unidades, precio y descuento. Calcula importe_bruto e importe_neto primero de forma exploratoria y luego mediante `preparar_productos`. Conserva el original y comprueba equivalencia. Se usa este nombre para no sobrescribir `preparar_ventas` de la parte guiada.


In [ ]:
productos = pd.DataFrame({
    "producto": ["A", "B", "A", "C"],
    "unidades": [2, 3, 1, 5],
    "precio": [10.0, 20.0, 15.0, 8.0],
    "descuento": [0.0, 0.10, 0.20, 1.0],
})
productos_originales = productos.copy(deep=True)
productos_exploratorios = productos.copy()
productos_exploratorios["importe_bruto"] = productos_exploratorios["unidades"] * productos_exploratorios["precio"]
productos_exploratorios["importe_neto"] = productos_exploratorios["importe_bruto"] * (1 - productos_exploratorios["descuento"])

def preparar_productos(datos: pd.DataFrame) -> pd.DataFrame:
    """Calcula los importes de cada operación sin modificar la entrada."""
    salida = datos.copy()
    salida["importe_bruto"] = salida["unidades"] * salida["precio"]
    salida["importe_neto"] = salida["importe_bruto"] * (1 - salida["descuento"])
    return salida

productos_preparados = preparar_productos(productos)
pd.testing.assert_frame_equal(productos_preparados, productos_exploratorios)
pd.testing.assert_frame_equal(productos, productos_originales)
np.testing.assert_allclose(productos_preparados["importe_neto"], [20, 54, 12, 0])
print(productos_preparados)


### Solución razonada

La función conserva el comportamiento del cálculo exploratorio y evita efectos secundarios. Un descuento de 1 equivale al 100 % y produce un neto de cero.


## Ejercicio 3 — Validación

Implementa `validar_productos`: columnas obligatorias, unidades y precio positivos, descuento entre 0 y 1 inclusive. Rechaza nulos y valores no numéricos o infinitos. Prueba entradas válidas y casos inválidos.


In [ ]:
def validar_productos(datos: pd.DataFrame) -> None:
    """Exige un esquema mínimo y valores finitos dentro de los rangos permitidos."""
    requeridas = {"producto", "unidades", "precio", "descuento"}
    faltantes = requeridas - set(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas: {sorted(faltantes)}")
    if datos[list(requeridas)].isna().any().any():
        raise ValueError("Las columnas obligatorias contienen valores nulos.")
    for columna in ["unidades", "precio", "descuento"]:
        if not pd.api.types.is_numeric_dtype(datos[columna]) or pd.api.types.is_bool_dtype(datos[columna]):
            raise ValueError(f"{columna} debe ser numérica y no booleana.")
        if not np.isfinite(datos[columna].to_numpy(dtype=float)).all():
            raise ValueError(f"{columna} contiene valores no finitos.")
    if (datos["unidades"] <= 0).any():
        raise ValueError("Las unidades deben ser positivas.")
    if (datos["precio"] <= 0).any():
        raise ValueError("El precio debe ser positivo.")
    if not datos["descuento"].between(0, 1, inclusive="both").all():
        raise ValueError("El descuento debe estar entre 0 y 1.")

validar_productos(productos)
casos_invalidos = {}
for nombre, columna, valor in [
    ("unidades cero", "unidades", 0),
    ("precio negativo", "precio", -1),
    ("descuento negativo", "descuento", -0.1),
    ("descuento mayor que uno", "descuento", 1.1),
    ("precio nulo", "precio", np.nan),
    ("precio infinito", "precio", np.inf),
]:
    caso = productos.copy()
    caso.loc[0, columna] = valor
    casos_invalidos[nombre] = caso
casos_invalidos["columna ausente"] = productos.drop(columns="precio")
casos_invalidos["precio texto"] = productos.assign(precio="texto")
casos_invalidos["unidades booleanas"] = productos.assign(unidades=True)
for nombre, caso in casos_invalidos.items():
    try:
        validar_productos(caso)
    except ValueError as error:
        print(f"{nombre}: {error}")
    else:
        raise AssertionError(f"La validación no detectó: {nombre}")
print("Entrada válida y casos inválidos comprobados.")


### Solución razonada

Comprobar el esquema primero evita un KeyError poco informativo. Los límites 0 y 1 son válidos para descuento. Los nulos y los infinitos requieren controles explícitos; una comparación de rango aislada puede dejar pasar algunos valores inválidos.


## Ejercicio 4 — Separar responsabilidades

Construye `validar_datos`, `preparar_datos` y `resumir_datos`. La primera valida, la segunda calcula los importes y la tercera agrega por producto. Compón las etapas sin modificar la entrada.


In [ ]:
def validar_datos(datos: pd.DataFrame) -> None:
    validar_productos(datos)

def preparar_datos(datos: pd.DataFrame) -> pd.DataFrame:
    return preparar_productos(datos)

def resumir_datos(datos: pd.DataFrame) -> pd.DataFrame:
    return (datos.groupby("producto", as_index=False)
            .agg(operaciones=("producto", "size"),
                 unidades=("unidades", "sum"),
                 importe_neto=("importe_neto", "sum")))

validar_datos(productos)
productos_pipeline = preparar_datos(productos)
resumen_productos = resumir_datos(productos_pipeline)
np.testing.assert_allclose(resumen_productos["importe_neto"].sum(), productos_pipeline["importe_neto"].sum())
pd.testing.assert_frame_equal(productos, productos_originales)
print(resumen_productos)


### Solución razonada

La composición es explícita. Las funciones de preparación y resumen no imprimen ni dibujan; el notebook presenta el resultado. Esto facilita reutilizarlas desde otros puntos de entrada.


## Ejercicio 5 — Visualización

Representa ventas netas por producto y la distribución del importe neto por operación. Añade una interpretación de 3–5 líneas para cada gráfico, basada en los datos del ejercicio.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(resumen_productos["producto"], resumen_productos["importe_neto"])
axes[0].set(title="Ventas netas por producto", xlabel="Producto", ylabel="Importe neto")
axes[1].hist(productos_pipeline["importe_neto"], bins=[0, 10, 20, 30, 40, 50, 60], edgecolor="white")
axes[1].set(title="Importe neto por operación", xlabel="Importe neto", ylabel="Operaciones")
plt.tight_layout()
plt.show()


### Solución razonada

**Barras — interpretación posible:**

El producto B acumula 54 de importe neto, frente a 32 de A y 0 de C.  
El producto C tiene ventas brutas, pero su descuento del 100 % anula el neto.  
Las cifras resumen cuatro operaciones; no describen una tendencia general.  
Para explicar las diferencias habría que revisar unidades, precios y descuentos.

**Histograma — interpretación posible:**

Los importes netos observados son 0, 12, 20 y 54.  
La distribución contiene una operación de importe superior a las demás.  
El cero se explica por el descuento total del producto C.  
Con cuatro observaciones, el histograma sirve para explorar, no para inferir una distribución poblacional.


## Ejercicio 6 — Reproducibilidad

Genera dos muestras con generadores sin semilla y dos muestras con generadores nuevos inicializados con la misma semilla. Explica por qué consumir dos veces un mismo generador avanza su estado.


In [ ]:
sin_semilla_1 = np.random.default_rng().normal(size=1000)
sin_semilla_2 = np.random.default_rng().normal(size=1000)
con_semilla_1 = np.random.default_rng(2026).normal(size=1000)
con_semilla_2 = np.random.default_rng(2026).normal(size=1000)
np.testing.assert_array_equal(con_semilla_1, con_semilla_2)
rng_avanza = np.random.default_rng(2026)
primera_extraccion = rng_avanza.normal(size=1000)
segunda_extraccion = rng_avanza.normal(size=1000)
print("Sin semilla, iguales:", np.array_equal(sin_semilla_1, sin_semilla_2))
print("Misma semilla y generadores nuevos, iguales:", np.array_equal(con_semilla_1, con_semilla_2))
print("Extracciones consecutivas de un generador, iguales:", np.array_equal(primera_extraccion, segunda_extraccion))


### Solución razonada

Reinicializar el generador reproduce la secuencia en el mismo entorno. Un generador conserva estado y cada extracción lo avanza. Sin semilla, las muestras normalmente difieren; no es una garantía lógica de desigualdad. Para reproducibilidad completa también hay que conservar versiones, datos, configuración y orden de ejecución.


## Ejercicio 7 — Benchmark

Compara el cálculo del importe neto mediante un bucle y una operación NumPy vectorizada. Valida equivalencia y realiza al menos diez mediciones de cada versión sobre la misma entrada. Representa los tiempos; no presupongas un factor fijo de aceleración.


In [ ]:
rng_bench = np.random.default_rng(7)
unidades_bench = rng_bench.integers(1, 10, 100_000)
precio_bench = rng_bench.uniform(1, 100, 100_000)
descuento_bench = rng_bench.uniform(0, 0.5, 100_000)

def neto_bucle(unidades, precio, descuento):
    return np.array([u * p * (1 - d) for u, p, d in zip(unidades, precio, descuento)])

def neto_vectorizado(unidades, precio, descuento):
    return unidades * precio * (1 - descuento)

argumentos_bench = (unidades_bench, precio_bench, descuento_bench)
np.testing.assert_allclose(neto_bucle(*argumentos_bench), neto_vectorizado(*argumentos_bench))
for funcion in [neto_bucle, neto_vectorizado]:
    funcion(*argumentos_bench)
mediciones = []
for repeticion in range(10):
    # Alternar el orden reduce un sesgo sencillo de la medición.
    metodos = [("Bucle", neto_bucle), ("Vectorizado", neto_vectorizado)]
    if repeticion % 2:
        metodos.reverse()
    for nombre, funcion in metodos:
        inicio = time.perf_counter()
        resultado_bench = funcion(*argumentos_bench)
        duracion = time.perf_counter() - inicio
        mediciones.append({"repeticion": repeticion + 1, "metodo": nombre, "segundos": duracion})
tiempos_practica = pd.DataFrame(mediciones)
assert tiempos_practica.groupby("metodo").size().eq(10).all()
print(tiempos_practica.groupby("metodo")["segundos"].agg(["median", "min", "max"]))
fig, ax = plt.subplots(figsize=(7, 4))
for nombre, grupo in tiempos_practica.groupby("metodo"):
    ax.plot(grupo["repeticion"], grupo["segundos"], marker="o", label=nombre)
ax.set(title="Diez mediciones por método", xlabel="Repetición", ylabel="Segundos")
ax.legend()
plt.show()


### Solución razonada

La comparación aísla la transformación: generación de datos y validación quedan fuera del tiempo medido. Ambas versiones devuelven un array. La mediana resume las repeticiones, pero los tiempos siguen dependiendo del equipo y del estado del sistema. Medir rendimiento no sustituye comprobar resultados.


## Ejercicio 8 — Estructura de proyecto

Propón una estructura para analizar churn a partir de CSV y ejecutar semanalmente un modelo. Explica qué pertenece a data, notebooks, src, tests y config. No hace falta entrenar un modelo en este ejercicio.


### Solución razonada

| Ruta | Responsabilidad |
|---|---|
| `data/raw/` | CSV de entrada conservados sin modificar; no incluir información sensible en Git |
| `data/processed/` | Datos derivados reproducibles |
| `notebooks/` | Exploración, gráficos e interpretación |
| `src/churn/loading.py` | Lectura de datos |
| `src/churn/validation.py` | Contratos de columnas y valores |
| `src/churn/features.py` | Transformaciones estables |
| `src/churn/analysis.py` | Resúmenes analíticos |
| `src/churn/pipeline.py` | Composición del flujo y punto de entrada semanal |
| `tests/` | Casos de validación y regresión de resultados |
| `config/` | Parámetros versionables sin secretos |
| `pyproject.toml` | Dependencias y configuración de herramientas |
| `.env.example` | Nombres de variables de entorno sin credenciales |
| `README.md` | Instalación, ejecución y contrato de salida |

El programador de tareas debe llamar al punto de entrada del pipeline con configuración explícita. El notebook permanece disponible para explorar y comunicar; no tiene que actuar como planificador semanal.


## Ejercicio 9 — Del notebook a src

Elige tres funciones de los ejercicios y crea módulos importables con ellas. Importa las funciones y compara su resultado con la versión del notebook. Evita sobrescribir archivos ajenos: utiliza una carpeta demo específica.


In [ ]:
import inspect
import importlib
import sys

raiz_modulos = Path("proyecto_productos_practica")
paquete = raiz_modulos / "src" / "productos_practica"
paquete.mkdir(parents=True, exist_ok=True)
(paquete / "__init__.py").write_text("", encoding="utf-8")
modulos = {
    "validation.py": "import numpy as np\nimport pandas as pd\n\n" + inspect.getsource(validar_productos),
    "features.py": "import pandas as pd\n\n" + inspect.getsource(preparar_productos),
    "analysis.py": "import pandas as pd\n\n" + inspect.getsource(resumir_datos),
}
for nombre, contenido in modulos.items():
    (paquete / nombre).write_text(contenido, encoding="utf-8")
ruta_src = str((raiz_modulos / "src").resolve())
if ruta_src not in sys.path:
    sys.path.insert(0, ruta_src)
importlib.invalidate_caches()
from productos_practica import validation, features, analysis
for modulo in [validation, features, analysis]:
    importlib.reload(modulo)
validation.validar_productos(productos)
preparados_modulo = features.preparar_productos(productos)
resumen_modulo = analysis.resumir_datos(preparados_modulo)
pd.testing.assert_frame_equal(preparados_modulo, productos_preparados)
pd.testing.assert_frame_equal(resumen_modulo, resumen_productos)
print("Importaciones y equivalencia verificadas:", paquete.resolve())


### Solución razonada

Validación, features y análisis quedan en módulos diferentes porque tienen responsabilidades distintas. `inspect.getsource` extrae las definiciones de las funciones para esta demostración. El ajuste de `sys.path` es didáctico; un proyecto instalable debería declarar el paquete e instalarse con `pip install -e .`.


## Ejercicio 10 — Reto final

Genera al menos 1.000 observaciones y cinco variables. Construye carga, validación, preparación y análisis con al menos tres funciones. Crea dos gráficos, compara con el análisis inicial y explica qué moverías a src. Utiliza nombres propios del reto para no alterar ejercicios anteriores.


In [ ]:
def cargar_reto(n: int = 1500, semilla: int = 2026) -> pd.DataFrame:
    generador = np.random.default_rng(semilla)
    return pd.DataFrame({
        "operacion_id": np.arange(1, n + 1),
        "producto": generador.choice(["A", "B", "C", "D"], n),
        "unidades": generador.integers(1, 10, n),
        "precio": generador.uniform(5, 100, n),
        "descuento": generador.choice([0.0, 0.1, 0.2], n),
    })

def validar_reto(datos: pd.DataFrame) -> None:
    validar_productos(datos)
    if "operacion_id" not in datos:
        raise ValueError("Falta operacion_id.")
    if datos["operacion_id"].isna().any() or datos["operacion_id"].duplicated().any():
        raise ValueError("operacion_id debe ser único y no nulo.")

def preparar_reto(datos: pd.DataFrame) -> pd.DataFrame:
    return preparar_productos(datos)

def analizar_reto(datos: pd.DataFrame) -> pd.DataFrame:
    return resumir_datos(datos)

def pipeline_reto(datos: pd.DataFrame):
    validar_reto(datos)
    preparados = preparar_reto(datos)
    return preparados, analizar_reto(preparados)

entrada_reto = cargar_reto()
entrada_reto_copia = entrada_reto.copy(deep=True)
# Referencia inicial independiente: cálculo por operación y agregación directa.
referencia_reto = entrada_reto.copy()
referencia_reto["importe_neto"] = [u * p * (1 - d) for u, p, d in zip(
    referencia_reto["unidades"], referencia_reto["precio"], referencia_reto["descuento"])]
resumen_referencia_reto = referencia_reto.groupby("producto")["importe_neto"].sum().sort_index()
preparados_reto, resumen_reto = pipeline_reto(entrada_reto)
pd.testing.assert_series_equal(
    resumen_referencia_reto,
    resumen_reto.set_index("producto")["importe_neto"].sort_index(),
)
pd.testing.assert_frame_equal(entrada_reto, entrada_reto_copia)
pd.testing.assert_frame_equal(cargar_reto(), entrada_reto)
assert len(entrada_reto) >= 1000 and entrada_reto.shape[1] >= 5
np.testing.assert_allclose(resumen_reto["importe_neto"].sum(), preparados_reto["importe_neto"].sum())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(resumen_reto["producto"], resumen_reto["importe_neto"])
axes[0].set(title="Reto: importe neto por producto", xlabel="Producto", ylabel="Importe neto")
axes[1].hist(preparados_reto["importe_neto"], bins=30, edgecolor="white")
axes[1].set(title="Reto: importe neto por operación", xlabel="Importe neto", ylabel="Operaciones")
plt.tight_layout()
plt.show()
print(resumen_reto)
print("Reto verificado: equivalencia, reproducibilidad y ausencia de mutación.")


### Solución razonada

**Interpretación:** las barras comparan el importe neto agregado, que depende tanto del número de operaciones como de sus unidades, precios y descuentos. El histograma muestra la variación del importe de cada operación. Los productos se asignaron aleatoriamente: las diferencias no prueban un efecto causal del producto. Los resultados son sintéticos y no representan el comportamiento de clientes reales.

**Separación propuesta:** mover `cargar_reto` a `loading.py`, `validar_reto` a `validation.py`, `preparar_reto` a `features.py`, `analizar_reto` a `analysis.py` y `pipeline_reto` a `pipeline.py`. Mantener los gráficos, la comparación y la interpretación en el notebook. Aquí la carga genera datos; en producción podría leer un CSV manteniendo el mismo contrato.


# 25. Preguntas de reflexión


1. ¿Cuándo deja de ser suficiente un notebook?
2. ¿Qué riesgo tiene reutilizar variables globales?
3. ¿Por qué una función mejora la testabilidad?
4. ¿Qué tipo de lógica debería moverse fuera del notebook?
5. ¿Por qué `Run All` es una prueba sencilla pero importante?
6. ¿Qué diferencias existen entre exploración y producción?
7. ¿Refactorizar implica necesariamente mejorar rendimiento?
8. ¿Puede un notebook muy largo seguir siendo profesional? Justifica tu respuesta.

## Respuestas orientativas

1. Cuando requiere ejecución recurrente, mantenimiento compartido o lógica crítica reutilizada, conviene complementar el notebook con módulos y un punto de entrada explícito.
2. Una variable global introduce dependencia del estado del kernel y puede cambiar resultados sin que la celda muestre su origen.
3. Una función con entradas y salida explícitas permite comprobar casos normales y límite de forma aislada.
4. Validación, transformación, reglas de negocio, carga y exportación reutilizadas deberían vivir en módulos.
5. Run All desde un kernel limpio detecta dependencias ocultas y errores de orden, pero no demuestra por sí solo que los resultados sean correctos.
6. La exploración favorece iteración y descubrimiento; la ejecución operativa necesita contratos, configuración, trazabilidad y pruebas.
7. No. Refactorizar conserva el comportamiento y mejora la estructura; una mejora de rendimiento debe medirse.
8. Sí, si mantiene un recorrido claro y reproducible. La longitud debe justificarse por el contenido y no por duplicaciones o lógica crítica dispersa.
